# Macroeconomic Time Series Analysis

## Overview

This notebook investigates the dynamic effects of U.S. monetary policy on real economic activity and inflation using a Structural Vector Autoregression (SVAR).

The analysis uses quarterly U.S. macroeconomic data on:

- Real GDP (GDPC1)
- GDP Price Index (GDPCTPI)
- Federal Funds Rate (FEDFUNDS)

A recursive (Cholesky) identification scheme is employed to identify monetary policy shocks and estimate their dynamic effects through impulse response functions.

The workflow covers:

- data preparation,
- variable transformation,
- lag selection,
- VAR estimation,
- model diagnostics,
- structural identification,
- impulse response analysis,
- and robustness checks through cross-validation.

In [ ]:
required_packages <- c("vars")
to_install <- required_packages[!required_packages %in% installed.packages()[, "Package"]]
if (length(to_install) > 0) install.packages(to_install)
library(vars)

## Setup

Import the macroeconomic datasets and prepare the variables required for the empirical analysis.

The dataset contains:

- Real GDP
- GDP Price Index
- Federal Funds Rate

All series are converted into a common quarterly frequency before estimation.

In [ ]:
gdp <- read.csv("GDPC1.csv")     
pi  <- read.csv("GDPCTPI.csv")   
ff  <- read.csv("FEDFUNDS.csv")  

# 1. Data Preparation

Merge the three macroeconomic series into a single dataset and construct the variables used throughout the analysis.

This step ensures that all observations share the same quarterly time index before estimating the econometric models.

In [ ]:
gdp$observation_date <- as.Date(gdp$observation_date)
pi$observation_date  <- as.Date(pi$observation_date)
ff$observation_date  <- as.Date(ff$observation_date)

ff <- ff[!is.na(ff$FEDFUNDS), ]  # drop incomplete current quarter, if present

df <- merge(gdp, pi, by = "observation_date")
df <- merge(df, ff, by = "observation_date")
df <- df[order(df$observation_date), ]
rownames(df) <- NULL

cat("Common sample:", as.character(min(df$observation_date)), "to",
    as.character(max(df$observation_date)), "| N =", nrow(df), "\n")

# 2. Variable Construction

Construct the transformed macroeconomic variables required for time series analysis.

Following standard macroeconomic practice:

- Real GDP is transformed into quarterly growth rates.
- Inflation is computed from the GDP Price Index.
- The Federal Funds Rate is kept in levels.

In [ ]:
df$Y1_gdp_growth <- c(NA, diff(log(df$GDPC1)))   * 100
df$Y2_inflation  <- c(NA, diff(log(df$GDPCTPI))) * 100
df$Y3_ffr        <- df$FEDFUNDS

df <- df[-1, ]  # drop first row (lost to differencing)
rownames(df) <- NULL

# 3. Time Series Object

Convert the prepared dataset into a multivariate time series object.

This object will be used for lag selection, VAR estimation, diagnostic testing, and impulse response analysis.

In [ ]:
start_year <- as.numeric(format(df$observation_date[1], "%Y"))
start_qtr  <- (as.numeric(format(df$observation_date[1], "%m")) - 1) %/% 3 + 1

y <- ts(df[, c("Y1_gdp_growth", "Y2_inflation", "Y3_ffr")],
        start = c(start_year, start_qtr), frequency = 4)
colnames(y) <- c("GDP growth", "Inflation", "FFR")

plot(y, main = "Transformed series")

# 4. Lag Order Selection

Selecting an appropriate lag length is a crucial step before estimating the VAR model.

Lag orders between one and three quarters are evaluated using the Bayesian Information Criterion (BIC).

As an additional robustness check, a rolling-origin cross-validation exercise compares the out-of-sample forecasting performance of the competing lag specifications.

In [ ]:
lagsel <- VARselect(y, lag.max = 3, type = "const")
print(lagsel$criteria)
p_bic <- unname(lagsel$selection["SC(n)"])
cat("BIC-selected lag order: p =", p_bic, "\n")

# 5. VAR Model Estimation

Estimate the Vector Autoregression using the selected lag order.

The VAR framework captures the dynamic interactions between output growth, inflation, and the policy interest rate by allowing each variable to depend on its own past values as well as the lagged values of the remaining variables.

In [ ]:
var_model <- VAR(y, p = p_bic, type = "const")
summary(var_model)

## Results

Both model-selection procedures identify a VAR(1) specification as the preferred model.

- BIC selects one lag among the candidate specifications.
- Rolling-origin cross-validation also favors one lag based on the lowest out-of-sample forecast error.

The agreement between statistical information criteria and predictive performance suggests that lag-order uncertainty is unlikely to affect the subsequent structural analysis.

# 6. Model Diagnostics

Before interpreting structural shocks, the estimated VAR model must satisfy its main statistical assumptions.

Two diagnostic checks are performed:

- Stability of the VAR system
- Residual serial correlation

The estimated VAR(1) satisfies the stability condition, with all characteristic roots lying inside the unit circle.

However, the Portmanteau test rejects the null hypothesis of white-noise residuals, suggesting that some short-run dynamics remain unexplained. Because the assignment restricts the lag search to a maximum of three lags, this limitation is acknowledged rather than corrected.

In [ ]:
cat("Moduli of characteristic roots (should all be < 1 for stability):\n")
print(roots(var_model))

print(serial.test(var_model, lags.pt = 8, type = "PT.asymptotic"))

## Diagnostic Assessment

Overall, the estimated VAR satisfies the stability requirement, allowing impulse response analysis to proceed.

Residual autocorrelation remains statistically significant, indicating that the model does not fully capture all short-run dynamics. Consequently, the empirical results should be interpreted with appropriate caution.

# 7. Structural Identification

To recover economically meaningful monetary policy shocks, a recursive (Cholesky) identification strategy is adopted.

The ordering is:

1. GDP Growth
2. Inflation
3. Federal Funds Rate

This recursive structure assumes that:

- GDP growth may contemporaneously affect inflation and monetary policy.
- Inflation may contemporaneously affect monetary policy.
- Monetary policy affects output and inflation only with a one-quarter delay.

This identification strategy follows the standard recursive monetary-policy SVAR framework commonly used in empirical macroeconomics.

In [ ]:
irf_bic <- irf(var_model, n.ahead = 20, ortho = TRUE,
               boot = TRUE, ci = 0.95, runs = 500, seed = 123)

# 8. Impulse Response Analysis

Impulse Response Functions (IRFs) describe how each macroeconomic variable reacts over time following an unexpected structural shock.

The responses are computed over a 20-quarter horizon using bootstrap confidence intervals.

These dynamic responses provide the main economic interpretation of the estimated VAR model.

In [ ]:
plot_irf_grid <- function(irf_obj, filename, col_line) {
  shocks <- names(irf_obj$irf)
  resps  <- colnames(irf_obj$irf[[1]])
  h      <- 0:(nrow(irf_obj$irf[[1]]) - 1)
  
  png(filename, width = 1100, height = 1100, res = 130)
  par(mfrow = c(3, 3), mar = c(4, 4, 3, 1))
  for (shock in shocks) {
    for (resp in resps) {
      point <- irf_obj$irf[[shock]][, resp]
      lower <- irf_obj$Lower[[shock]][, resp]
      upper <- irf_obj$Upper[[shock]][, resp]
      plot(h, point, type = "l", lwd = 2, col = col_line,
           ylim = range(c(lower, upper, 0)),
           xlab = "Quarters", ylab = resp,
           main = paste0("Shock: ", shock))
      lines(h, lower, lty = 2, col = "grey40")
      lines(h, upper, lty = 2, col = "grey40")
      abline(h = 0, col = "red", lty = 3)
    }
  }
  dev.off()
}

In [ ]:
plot_irf_grid(irf_bic, "IRF_3x3_BIC_model.png", "steelblue4")
cat("Saved IRF_3x3_BIC_model.png\n")

## Main Findings

The impulse response analysis provides three main findings.

- Positive GDP shocks are followed by increases in the Federal Funds Rate, suggesting a systematic monetary policy response to stronger economic activity.

- Inflation shocks generate persistent increases in inflation and induce gradual monetary tightening.

- Contractionary monetary policy shocks produce a modest negative response of output growth, while inflation exhibits the well-known "price puzzle", with inflation increasing temporarily following a policy tightening.

Overall, the estimated responses are broadly consistent with the existing SVAR literature and highlight both the strengths and limitations of recursive identification.

# 9. Robustness Check: Cross-Validation

Although the Bayesian Information Criterion selects the preferred lag order, an additional rolling-origin cross-validation procedure is performed.

The objective is to verify that the selected specification also provides the best out-of-sample forecasting performance, increasing confidence in the empirical results.

In [ ]:
cv_lag_selection <- function(y, p_grid = 1:3, n_init = 80) {
  Tobs <- nrow(y)
  cv_sse <- matrix(NA, nrow = Tobs - n_init, ncol = length(p_grid))
  colnames(cv_sse) <- paste0("p", p_grid)
  
  for (j in seq_along(p_grid)) {
    p <- p_grid[j]
    for (t in n_init:(Tobs - 1)) {
      train <- y[1:t, ]
      mod <- tryCatch(VAR(train, p = p, type = "const"), error = function(e) NULL)
      if (is.null(mod)) next
      fc     <- predict(mod, n.ahead = 1)
      actual <- y[t + 1, ]
      pred   <- sapply(fc$fcst, function(x) x[1, "fcst"])
      err    <- actual - pred
      cv_sse[t - n_init + 1, j] <- sum(err^2)
    }
  }
  colMeans(cv_sse, na.rm = TRUE)
}

In [ ]:
cv_mse <- cv_lag_selection(y, p_grid = 1:3, n_init = 80)
cat("Cross-validation mean squared forecast error by lag order:\n")
print(cv_mse)

p_cv <- as.numeric(sub("p", "", names(which.min(cv_mse))))
cat("CV-selected lag order: p =", p_cv, "\n")

if (p_cv != p_bic) {
  cat("CV and BIC disagree (BIC p =", p_bic, ", CV p =", p_cv,
      ") -- estimating the alternative model for comparison.\n")
  var_model_cv <- VAR(y, p = p_cv, type = "const")
  irf_cv <- irf(var_model_cv, n.ahead = 20, ortho = TRUE,
                boot = TRUE, ci = 0.95, runs = 500, seed = 123)
  plot_irf_grid(irf_cv, "IRF_3x3_CV_model.png", "darkorange3")
  cat("Saved IRF_3x3_CV_model.png -- compare visually with the BIC version.\n")
} else {
  cat("CV agrees with BIC on p =", p_bic, "-- no second model needed.\n")
}

## Robustness Assessment

Cross-validation confirms the lag order selected by the Bayesian Information Criterion.

Since both approaches independently select the same specification, the impulse response analysis does not appear to be sensitive to lag-order selection within the candidate models considered.

# Final Remarks

This analysis finds that U.S. monetary policy responds systematically to macroeconomic conditions, while the estimated effects of monetary policy shocks on output and inflation are comparatively weaker.

The recursive SVAR identifies the expected contractionary effect on output, although statistical significance is limited. Inflation exhibits the well-known price puzzle, a common finding in recursively identified monetary-policy VAR models.

The agreement between information criteria and cross-validation supports the selected VAR specification, while the diagnostic tests highlight the limitations of the constant-parameter recursive framework.

Overall, the notebook illustrates a complete empirical workflow for structural macroeconomic time series analysis, combining reproducible data preparation, econometric modeling, structural identification, and robustness analysis.